# Crypto Dataset Cleaning
This notebook documents and reproduces the complete data cleaning, sentiment integration, and feature engineering pipeline for cryptocurrency OHLCV price data. It combines historical price data with external news sentiment information from `cryptonews.csv`, which contains timestamped JSON `Sentiment` column.

### Goal
- Clean each asset CSV
    + Convert the Date column to proper datetime format
    + Convert OHLCV columns (Open, High, Low, Close, Volume) to numeric types
    + Remove duplicate rows based on date
    + Sort data
    + Remove invalid or inconsistent rows, such as:
        - Missing price values
        - Invalid OHLC relationships (e.g., Low > High)
        - Zero-liquidity rows (no price movement and zero volume)
- Build daily sentiment features from news:
    + `sentiment_score_mean` (mapped from class: positive=+1, neutral=0, negative=-1): The daily mean provides an overall sentiment direction signal.
    + `sentiment_polarity_mean`: Measures the strength and direction of sentiment.
    + `sentiment_subjectivity_mean`: Measures how subjective or opinion-based the news is.
    + `news_count`: The number of news articles per day
- Merge sentiment data with daily price data
    + Price data before the first news date is removed to ensure that all observations used for modeling have sentiment feature.
    + Merge sentiment features with price data using the daily date
    + Forward-fill sentiment only inside the news window
        - Forward-filling is used because sentiment effects often persist beyond the exact publication time of news. This allows the model to capture          delayed market reactions and avoids gaps in the sentiment signal.
    + Leave sentiment blank after the last news date.
- Feature engineering
    + return: Daily percentage price change, representing short-term price movement.
    + log_return: Logarithmic return
    + vol_7d and vol_30d: Rolling volatility over 7-day and 30-day windows.
    + ma_7 and ma_30: Rolling moving averages over 7-day and 30-day windows.
    + ma_ratio: Ratio between short-term and long-term moving averages


## 1) Imports and configuration


In [1]:
import os
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)

# ===== CONFIG =====
DATA_DIR = "data"

RAW_DIR = os.path.join(DATA_DIR, "raw")
TOKEN_DIR = os.path.join(RAW_DIR, "token_datasets")
SEMANTIC_DIR = os.path.join(RAW_DIR, "semantic")

OUTPUT_DIR = os.path.join(DATA_DIR, "clean")

NEWS_FILEPATH = os.path.join(SEMANTIC_DIR, "cryptonews.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

NUMERIC_COLS = ["Open", "High", "Low", "Close", "Volume"]


## 2) Load and inspect one raw asset file (example)


In [2]:
# Example: load a single asset file to see what it looks like.
# Change this filename to any file in INPUT_DIR, e.g. "bitcoin.csv".
example_asset_path = os.path.join(TOKEN_DIR, "bitcoin.csv")

if os.path.exists(example_asset_path):
    raw = pd.read_csv(example_asset_path)
    display(raw.head(10))
    print("Shape:", raw.shape)
else:
    print("Example asset file not found:", example_asset_path)
    print("Put your raw CSVs into:", TOKEN_DIR)


,Date,Close,High,Low,Open,Volume
0,NaN,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
1,2014-09-17,457.3340148925781,468.17401123046875,452.4219970703125,465.864013671875,21056800
2,2014-09-18,424.44000244140625,456.8599853515625,413.10400390625,456.8599853515625,34483200
3,2014-09-19,394.7959899902344,427.8349914550781,384.5320129394531,424.1029968261719,37919700
4,2014-09-20,408.90399169921875,423.2959899902344,389.88299560546875,394.6730041503906,36863600
5,2014-09-21,398.8210144042969,412.4259948730469,393.1809997558594,408.0849914550781,26580100
6,2014-09-22,402.1520080566406,406.9159851074219,397.1300048828125,399.1000061035156,24127600
7,2014-09-23,435.7909851074219,441.5570068359375,396.1969909667969,402.0920104980469,45099500
8,2014-09-24,423.2049865722656,436.11199951171875,421.1319885253906,435.7510070800781,30627700
9,2014-09-25,411.5740051269531,423.5199890136719,409.4679870605469,423.156005859375,26814400


Shape: (4130, 6)


## 3) Load and inspect cryptonews.csv


In [3]:
news = pd.read_csv(NEWS_FILEPATH)
display(news.head(10))
print("Shape:", news.shape)
print("Columns:", list(news.columns))


,Date,Sentiment,source,subject,text,title,url
0,10/12/2021 20:00,"{'class': 'positive', 'polarity': 0.16, 'subje...",CryptoNews,blockchain,"Within a little more than a year, Celo aims to...","Celo to Be Fastest EVM Chain by End of 2022, C...",https://cryptonews.com/news/celo-to-be-fastest...
1,10/15/2021 0:00,"{'class': 'neutral', 'polarity': 0.0, 'subject...",CryptoNews,blockchain,Chinese companies are still topping the blockc...,Tech Crackdown Hasn't Halted Chinese Firms' Bl...,https://cryptonews.com/news/tech-crackdown-has...
2,10/18/2021 13:58,"{'class': 'positive', 'polarity': 0.14, 'subje...",CryptoNews,blockchain,Advancing its project to become \x9caÂ\xa0meta...,"Facebook To Add 10,000 Jobs In EU For Metavers...",https://cryptonews.com/news/facebook-to-add-10...
3,10/19/2021 13:39,"{'class': 'positive', 'polarity': 0.1, 'subjec...",CryptoNews,blockchain,Banque de France disclosed the results of its ...,French Central Bank's Blockchain Bond Trial Br...,https://cryptonews.com/news/french-central-ban...
4,10/27/2021 15:17,"{'class': 'neutral', 'polarity': 0.0, 'subject...",CryptoNews,defi,Cream Finance (CREAM) suffered another flash l...,Cream Finance Suffers Another Exploit as Attac...,https://cryptonews.com/news/cream-finance-suff...
5,10/29/2021 10:40,"{'class': 'positive', 'polarity': 0.2, 'subjec...",CryptoNews,defi,The crypto community has issued a withering re...,FATF Wants to 'Gut' DeFi with 'Vague' New Guid...,https://cryptonews.com/news/fatf-wants-to-gut-...
6,11/1/2021 15:18,"{'class': 'neutral', 'polarity': 0.0, 'subject...",CryptoNews,defi,'This is finally getting to the point where cr...,Google's Parent Increases its Crypto Bet by Jo...,https://cryptonews.com/news/google-increases-i...
7,11/2/2021 15:07,"{'class': 'negative', 'polarity': -0.2, 'subje...",CryptoNews,nft,'Each NFT at auction contains 'secret' content...,SCRT Rallies As Quentin Tarantino Releases NFT...,https://cryptonews.com/news/scrt-rallies-as-qu...
8,11/2/2021 19:00,"{'class': 'positive', 'polarity': 0.15, 'subje...",CryptoNews,nft,"The buyer, confronting an over 99% discount on...",CryptoPunk Mistakenly Sells at Over 99% Discou...,https://cryptonews.com/news/cryptopunk-mistake...
9,11/3/2021 10:41,"{'class': 'neutral', 'polarity': 0.0, 'subject...",CryptoNews,ethereum,Spot bitcoin ETF is 'also possible in 2022.',Ethereum Futures ETF May Come Before Spot Bitc...,https://cryptonews.com/news/ethereum-futures-e...


Shape: (31037, 7)
Columns: ['Date', 'Sentiment', 'source', 'subject', 'text', 'title', 'url']


## 4) Convert cryptonews sentiment JSON into daily features


In [4]:
def load_daily_sentiment(filepath):

    news = pd.read_csv(filepath)
    news["Date"] = pd.to_datetime(news["Date"], errors="coerce")
    news = news.dropna(subset=["Date"])

    # Time alignment
    # The news dataset contains timestamps with specific times (e.g., 2021-10-12 20:00) while price data is recorded at daily frequency
    # We need to align them
    news["Date"] = news["Date"].dt.floor("D")

    # Parse JSON safely
    def parse_sentiment(x):
        try:
            if isinstance(x, str):
                return json.loads(x.replace("'", '"'))
            return {}
        except:
            return {}

    sentiment = news["Sentiment"].apply(parse_sentiment).apply(pd.Series)

    # Convert class to numeric score
    class_map = {
        "positive": 1,
        "neutral": 0,
        "negative": -1
    }

    sentiment["score"] = sentiment["class"].map(class_map)

    sentiment["polarity"] = pd.to_numeric(sentiment["polarity"], errors="coerce")
    sentiment["subjectivity"] = pd.to_numeric(sentiment["subjectivity"], errors="coerce")

    expanded = pd.concat([news[["Date", "text"]], sentiment], axis=1)

    # Aggregate per day
    daily = (
        expanded.groupby("Date")
        .agg(
            sentiment_score_mean=("score", "mean"),
            sentiment_polarity_mean=("polarity", "mean"),
            sentiment_subjectivity_mean=("subjectivity", "mean"),
            news_count=("score", "count"),
            news_text=("text", lambda x: ".".join(x.dropna()))

        )
        .reset_index()
    )

    return daily

## 5) Main cleaning


In [ ]:
DAILY_SENTIMENT = load_daily_sentiment(NEWS_FILEPATH)

FIRST_NEWS_DATE = DAILY_SENTIMENT["Date"].min()
LAST_NEWS_DATE = DAILY_SENTIMENT["Date"].max()

SENTIMENT_COLS = [
    "sentiment_score_mean",
    "sentiment_polarity_mean",
    "sentiment_subjectivity_mean",
    "news_count"
]

def clean_crypto_file(input_path, output_path):
    asset = os.path.splitext(os.path.basename(input_path))[0].upper()
    df = pd.read_csv(input_path)

    # Convert blank / whitespace cells to NaN
    df = df.replace(r'^\s*$', np.nan, regex=True)

    # Drop rows where Date is missing or not a real date
    df = df[df["Date"].notna()]

    # Convert Date to datetime (invalid ones become NaT)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df[df["Date"].notna()]

    # Convert numeric columns
    for col in NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in NUMERIC_COLS:
        # Convert empty strings or invalid to NaN
        df[col] = df[col].replace("", np.nan)

        # Fill NaN with median of that column
        median_value = df[col].median()

        df[col] = df[col].fillna(median_value)

    # Drop rows where all price columns are missing
    df = df.dropna(subset=["Open", "High", "Low", "Close"], how="all")

    # ---------- Remove duplicates & sort ----------
    df = df.drop_duplicates(subset="Date")
    df = df.sort_values("Date").reset_index(drop=True)

    # ---------- OHLC consistency ----------
    bad_ohlc = (
        (df["Low"] > df["High"]) |
        (df["Open"] < df["Low"]) | (df["Open"] > df["High"]) |
        (df["Close"] < df["Low"]) | (df["Close"] > df["High"])
    )
    df = df[~bad_ohlc]

    # ---------- Remove zero-liquidity rows ----------
    df = df[~((df["Volume"] == 0) & (df["Open"] == df["Close"]))]

    # ---------- Merge sentiment ----------
    df = df.merge(DAILY_SENTIMENT, on="Date", how="inner")

    # ---------- Feature engineering ----------
    df["return"] = df["Close"].pct_change()
    df["log_return"] = np.log(df["Close"] / df["Close"].shift(1))

    df["vol_7d"] = df["log_return"].rolling(7).std()
    df["vol_30d"] = df["log_return"].rolling(30).std()

    df["ma_7"] = df["Close"].rolling(7).mean()
    df["ma_30"] = df["Close"].rolling(30).mean()
    df["ma_ratio"] = df["ma_7"] / df["ma_30"]

    df["return"] = df["return"].fillna(0)
    df["log_return"] = df["log_return"].fillna(0)
    
    df["vol_7d"] = df["vol_7d"].fillna(0)
    df["vol_30d"] = df["vol_30d"].fillna(0)
    
    df["ma_7"] = df["ma_7"].fillna(0)
    df["ma_30"] = df["ma_30"].fillna(0)
    
    df["ma_ratio"] = df["ma_ratio"].fillna(0)


    # ---------- Labels  ----------
    df["next_close"] = df["Close"].shift(-1)

    # 1 if next day's close is higher than today's close, else 0
    df["price_increase"] = (df["next_close"] > df["Close"]).astype("Int64")

    # Set last row to nan explicitly (since next_close is NA)
    df.loc[df["next_close"].isna(), "price_increase"] = pd.nan
    
    df = df.reset_index(drop=True)

    # Save cleaned file
    df.to_csv(output_path, index=False)

    print(f"Cleaned: {asset}")

## 6) Process all asset files


In [6]:
# ===== RUN FOR ALL FILES =====
for file in os.listdir(TOKEN_DIR):

    if file.endswith(".csv"):

        input_path = os.path.join(TOKEN_DIR, file)
        output_path = os.path.join(OUTPUT_DIR, file)

        clean_crypto_file(input_path, output_path)

Cleaned: AAVE
Cleaned: ALGORAND
Cleaned: APTOS
Cleaned: ARBITRUM
Cleaned: AVALANCHE
Cleaned: AXIE_INFINITY
Cleaned: BINANCE_COIN
Cleaned: BITCOIN
Cleaned: BITCOIN_CASH
Cleaned: CARDANO
Cleaned: CHAINLINK
Cleaned: COSMOS
Cleaned: DECENTRALAND
Cleaned: DOGECOIN
Cleaned: EOS
Cleaned: ETHEREUM
Cleaned: FANTOM
Cleaned: FILECOIN
Cleaned: FLOW
Cleaned: HEDERA
Cleaned: IMMUTABLE
Cleaned: INJECTIVE
Cleaned: INTERNET_COMPUTER
Cleaned: KASPA
Cleaned: LIDO
Cleaned: LITECOIN
Cleaned: MAKER
Cleaned: NEAR
Cleaned: OPTIMISM
Cleaned: PEPE
Cleaned: POLKADOT
Cleaned: POLYGON
Cleaned: RENDER
Cleaned: SANDBOX
Cleaned: SHIBA_INU
Cleaned: SOLANA
Cleaned: STACKS
Cleaned: STELLAR
Cleaned: SUI
Cleaned: TETHER
Cleaned: TEZOS
Cleaned: THETA
Cleaned: THE_GRAPH
Cleaned: TONCOIN
Cleaned: TRON
Cleaned: UNISWAP
Cleaned: USD_COIN
Cleaned: VECHAIN
Cleaned: XRP


### Output dataset structure

The final cleaned dataset contains the following types of features:

- Raw price features: Open, High, Low, Close, Volume
- Sentiment features: sentiment_score_mean, sentiment_polarity_mean, sentiment_subjectivity_mean, news_count
- Derived financial features: return, log_return, volatility measures, and moving average features

## 7) Inspect one cleaned file

In [7]:
example_cleaned_asset_path = os.path.join(OUTPUT_DIR, "bitcoin.csv")

if os.path.exists(example_cleaned_asset_path):
    cleaned = pd.read_csv(example_cleaned_asset_path)
    display(cleaned.head(10))
    print("Shape:", cleaned.shape)
else:
    print("Example asset file not found:", example_cleaned_asset_path)

,Date,Close,High,Low,Open,Volume,sentiment_score_mean,sentiment_polarity_mean,sentiment_subjectivity_mean,news_count,news_text,return,log_return,vol_7d,vol_30d,ma_7,ma_30,ma_ratio,next_close,price_increase
0,2021-10-12,56041.058594,57627.878906,54477.972656,57526.832031,41083758949,1.0,0.16,0.50,1.0,"Within a little more than a year, Celo aims to...",0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,57401.097656,1.0
1,2021-10-13,57401.097656,57688.660156,54370.972656,56038.257812,41684252783,1.0,0.16,0.50,1.0,NaN,0.024269,0.023979,0.000000,0.0,0.000000,0.0,0.0,57321.523438,0.0
2,2021-10-14,57321.523438,58478.734375,56957.074219,57372.832031,36615791366,1.0,0.16,0.50,1.0,NaN,-0.001386,-0.001387,0.000000,0.0,0.000000,0.0,0.0,61593.949219,1.0
3,2021-10-15,61593.949219,62757.128906,56868.144531,57345.902344,51780081801,0.0,0.00,0.00,1.0,Chinese companies are still topping the blockc...,0.074534,0.071887,0.000000,0.0,0.000000,0.0,0.0,60892.179688,0.0
4,2021-10-16,60892.179688,62274.476562,60206.121094,61609.527344,34250964237,0.0,0.00,0.00,1.0,NaN,-0.011393,-0.011459,0.000000,0.0,0.000000,0.0,0.0,61553.617188,1.0
5,2021-10-17,61553.617188,61645.523438,59164.468750,60887.652344,29032367511,0.0,0.00,0.00,1.0,NaN,0.010862,0.010804,0.000000,0.0,0.000000,0.0,0.0,62026.078125,1.0
6,2021-10-18,62026.078125,62614.660156,60012.757812,61548.804688,38055562075,1.0,0.14,0.45,1.0,Advancing its project to become \x9caÂ\xa0meta...,0.007676,0.007646,0.000000,0.0,59547.071987,0.0,0.0,64261.992188,1.0
7,2021-10-19,64261.992188,64434.535156,61622.933594,62043.164062,40471196346,1.0,0.10,0.40,1.0,Banque de France disclosed the results of its ...,0.036048,0.035413,0.027775,0.0,60721.491071,0.0,0.0,65992.835938,1.0
8,2021-10-20,65992.835938,66930.390625,63610.675781,64284.585938,40788955582,1.0,0.10,0.40,1.0,NaN,0.026934,0.026578,0.027861,0.0,61948.882254,0.0,0.0,62210.171875,0.0
9,2021-10-21,62210.171875,66600.546875,62117.410156,66002.234375,45908121370,1.0,0.10,0.40,1.0,NaN,-0.057319,-0.059028,0.040748,0.0,62647.260603,0.0,0.0,60692.265625,0.0


Shape: (1547, 20)
